#### Objetivo: Apresentar as métricas dos modelos dado os hiperparâmetros p,x_lag,real_degree,img_degree

In [1]:
# ---------------------
# Incluindo Bibliotecas
# ---------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.api import VARMAX
from statsmodels.tsa.stattools import coint,kpss
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
from itertools import product,combinations
import os

##### Lendo base de dados

In [16]:
def get_result_file_name(p:int,xl:int,r:int,i:int) -> str:
    fname = f"results/larger-dataset-test-results-p{p}-xl{xl}-r{r}-i{i}.csv"
    return fname

In [17]:
res_path = get_result_file_name(3,3,3,3)
df_res = pd.read_csv(res_path)

In [18]:
print(list(df_res.columns))
best_yr_param = pd.Series(df_res.sort_values(by="Yreal_RMSE").iloc[0])
best_yi_param = pd.Series(df_res.sort_values(by="Yimg_RMSE").iloc[0])

['Unnamed: 0', 'rmse_mean', 'aic', 'bic', 'p', 'x_lag', 'exog_degree', 'trend', 'Yreal_MSE', 'Yreal_RMSE', 'Yreal_MAE', 'Yreal_R2', 'Yreal_AdjR2', 'Yreal_STD', 'Yimg_MSE', 'Yimg_RMSE', 'Yimg_MAE', 'Yimg_R2', 'Yimg_AdjR2', 'Yimg_STD']


#### Visualizando Dados

In [24]:
# -----------------------------
# Função de Impressão dos Dados
# -----------------------------
def view_result(data,meta=None,endog_cols=('Yreal', 'Yimg')):
    """
    Imprime na tela as métricas calculadas de acordo com a previsão do modelo.
    Aceita `data` como dicionário ou DataFrame (métricas nas linhas, colunas endog nas colunas).
    `meta` é obrigatório quando `data` for um DataFrame: dict com p, aic, bic, rmse_mean, x_lag, exog_degree.
    """
    metric_labels = [
        ('RMSE',  'RMSE'),
        ('MAE',   'MAE'),
        ('R2',    'R²'),
        ('AdjR2', 'R² Ajust'),
        ('STD',   'STD')]

    W_METRIC = 10
    W_VAL    = 10

    def sep(l, m, r, n_cols):
        block = '─' * (W_METRIC + W_VAL + 3)
        return l + m.join([block] * n_cols) + r

    def row_str(pairs):
        cells = [f' {lbl:<{W_METRIC}} {val:>{W_VAL}} ' for lbl, val in pairs]
        return '│' + '│'.join(cells) + '│'

    is_df = isinstance(data, pd.DataFrame)

    def get_meta(key):
        return meta[key] if is_df else data[key]

    def get_metric(col, key):
        return data.at[key, col] if is_df else data[col][key]

    print(f'Lag={get_meta("p")}')
    print(f'  AIC={get_meta("aic"):.2f} | BIC={get_meta("bic"):.2f} | RMSE Mean={get_meta("rmse_mean"):.4f}')
    print(f'  X Lag={get_meta("x_lag")} | Exog Degree={get_meta("exog_degree")}')

    print(sep('  ┌', '┬', '┐', len(endog_cols)))
    print('  ' + row_str([(col, '') for col in endog_cols]))
    print(sep('  ├', '┼', '┤', len(endog_cols)))

    for key, lbl in metric_labels:
        pairs = []
        for col in endog_cols:
            val    = get_metric(col, key)
            v_str  = f'{val:+.4f}' if key in ('R2', 'AdjR2') else f'{val:.4f}'
            pairs.append((lbl, v_str))
        print('  ' + row_str(pairs))

    print(sep('  └', '┴', '┘', len(endog_cols)))

In [25]:
# ------------------------------------------------
# Função de Recuperação de Dados para Visualização
# ------------------------------------------------
def row_to_metrics_df(row:pd.Series,endog_cols:list=('Yreal', 'Yimg')) -> tuple[pd.DataFrame, dict]:
    """
    Reconstrói o df_metrics e meta a partir de uma linha do DataFrame de resultados.
    """
    metric_keys = ['MSE', 'RMSE', 'MAE', 'R2', 'AdjR2', 'STD']
    meta_keys   = ['p', 'x_lag', 'exog_degree', 'trend', 'aic', 'bic', 'rmse_mean']

    records = {}
    for col in endog_cols:
        records[col] = {m: row[f'{col}_{m}'] for m in metric_keys}

    df_metrics = pd.DataFrame(records)
    meta       = {k: row[k] for k in meta_keys}

    return df_metrics, meta

In [26]:
metrics, meta = row_to_metrics_df(best_yr_param)
view_result(metrics,meta=meta)

Lag=2
  AIC=919964.80 | BIC=920483.15 | RMSE Mean=0.5326
  X Lag=2 | Exog Degree=(r:3,i:3)
  ┌───────────────────────┬───────────────────────┐
  │ Yreal                 │ Yimg                  │
  ├───────────────────────┼───────────────────────┤
  │ RMSE           0.5321 │ RMSE           0.5332 │
  │ MAE            0.4400 │ MAE            0.4412 │
  │ R²            +0.9983 │ R²            +0.9983 │
  │ R² Ajust      +0.9983 │ R² Ajust      +0.9983 │
  │ STD            0.5321 │ STD            0.5331 │
  └───────────────────────┴───────────────────────┘


In [ ]:
metrics, meta = row_to_metrics_df(best_yi_param)
view_result(metrics,meta=meta)